In [ ]:
import logging
import csv
from typing import Any, Dict, List
import json
import json
import os
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
import difflib

load_dotenv()  # Load environment variables from .env file if present


from kinexon_handball_api.handball import HandballAPI

In [ ]:
# connect duckdb
import duckdb
con = duckdb.connect(database='../data/mydb2024-25.duckdb', read_only=False)

In [ ]:
api = HandballAPI(
    base_url=os.getenv(
        "ENDPOINT_KINEXON_SESSION", "https://hbl-cloud.kinexon.com/api"
    ),
    api_key=os.getenv("API_KEY_KINEXON", "your_api_key_here"),
    username_basic=os.getenv("USERNAME_KINEXON_SESSION", "your_username_here"),
    password_basic=os.getenv("PASSWORD_KINEXON_SESSION", "your_password_here"),
    username_main=os.getenv("USERNAME_KINEXON_MAIN", "your_username_here"),
    password_main=os.getenv("PASSWORD_KINEXON_MAIN", "your_password_here"),
    endpoint_session=os.getenv(
        "ENDPOINT_KINEXON_SESSION",
        "https://hbl-cloud.kinexon.com/api/session",
    ),
    endpoint_main=os.getenv(
        "ENDPOINT_KINEXON_MAIN",
        "https://hbl-cloud.kinexon.com/api",
    ),
    timeout=10000,
)

In [ ]:
# from duckdb, query all session_ids in table  kinexon_positions: ['ts in ms', 'formatted local time', 'sensor id', 'mapped id', 'number', 'full name', 'league id', 'group id', 'group name', 'x in m', 'y in m', 'speed in m/s', 'direction of movement in deg', 'acceleration in m/s2', 'total distance in m', 'metabolic power in W/kg', 'acceleration load', 'Unnamed: 17', 'session_id', 'fixture_id']
session_ids = con.execute("SELECT DISTINCT session_id FROM kinexon_positions ORDER BY session_id").fetchall()
session_ids = [s[0] for s in session_ids]
for session_id in session_ids:
    print(f"Syncing session_id: {session_id}")
    # fetch events for session_id
    events = api.get_events_for_session(session_id=session_id)
    events_dict = [event.to_dict() for event in events]
    print(f"Found {len(events)} events for session_id: {session_id}")
    # convert events to dataframe
    df_events_shot_detected = pd.DataFrame(events_dict)
    break

# first show excerpt of kinexon_positions for session_id
df_positions = con.execute(f"SELECT * FROM kinexon_positions WHERE session_id='{session_id}'").fetchdf()
display(df_positions)
# show events dataframe
display(df_events_shot_detected)




In [ ]:
# unsure about the keys, show columns of df_events
print(df_events_shot_detected.columns.tolist())
# df_events.columns = ['timestamp', 'timestamp_ms', 'timezone_id', 'game_clock', 'period', 'player_id', 'distance', 'speed_ball', 'trajectory', 'shot_position_x', 'shot_position_y', 'hit_position_y', 'hit_position_z', 'success', 'shot_category', 'goalkeeper_id', 'shot_type', 'assisting_player_id', 'validated', 'id', 'event_type', 'league_id']
# show columns of df_positions
print(df_positions.columns.tolist())
# df_positions.columns = ['ts in ms', 'formatted local time', 'sensor id', 'mapped id', 'number', 'full name', 'league id', 'group id', 'group name', 'x in m', 'y in m', 'speed in m/s', 'direction of movement in deg', 'acceleration in m/s2', 'total distance in m', 'metabolic power in W/kg', 'acceleration load', 'Unnamed: 17', 'session_id', 'fixture_id']

# in df_events, insert ts_in_positions which is the closest ts in ms in df_positions
df_events_shot_detected['ts_in_positions'] = df_events_shot_detected['timestamp_ms'].apply(
    lambda x: df_positions['ts in ms'].iloc[(df_positions['ts in ms'] - x).abs().argsort()[:1]].values[0]
)
# show df_events with new column before dropping 
display(df_events_shot_detected)
# drop events that happened before first timestamp in df_positions
df_events_shot_detected = df_events_shot_detected[df_events_shot_detected['ts_in_positions'] >= df_positions['ts in ms'].min()]
# drop events not validated
df_events_shot_detected = df_events_shot_detected[df_events_shot_detected['validated'] != 0]
display(df_events_shot_detected)

# join df_events and df_positions on mapped id = player_id and timestamp_ms = ts in ms
df_merged = pd.merge(
    df_events_shot_detected,
    df_positions,
    left_on=['player_id', 'ts_in_positions'],
    right_on=['mapped id', 'ts in ms'],
    how='left'
)

display(df_merged)

In [ ]:
df_shot_positions_detected_events = df_merged[['shot_position_x', 'shot_position_y', 'x in m', 'y in m']]
print("List of shot positions from detected events (shot_position_x, shot_position_y, x in m, y in m):")

# shot_position_x, shot_position_y has center in the middle of the game field
# x in m, y in m has center in the bottom left corner of the game field
df_shot_positions_detected_events['shot_position_x'] = df_shot_positions_detected_events['shot_position_x'].astype(float) + 20.0
df_shot_positions_detected_events['shot_position_y'] = 20 - (df_shot_positions_detected_events['shot_position_y'].astype(float) + 10.0)

for pos in df_shot_positions_detected_events.values.tolist():
    print(pos)

In [ ]:
# Nicely styled Plotly scatter of shot positions over a handball field
import base64
from pathlib import Path
import plotly.graph_objects as go

# --- config ---
bg_image_path = Path("../assets/handballfeld.png")
x_range = (0, 40)
y_range = (0, 20)

# --- encode background image so it renders reliably in browsers ---
bg_source = None
if bg_image_path.exists():
    bg_source = "data:image/png;base64," + base64.b64encode(bg_image_path.read_bytes()).decode("utf-8")

fig = go.Figure()

# primary trace
if {"shot_position_x", "shot_position_y"}.issubset(df_shot_positions_detected_events.columns):
    fig.add_trace(
        go.Scattergl(
            x=df_shot_positions_detected_events["shot_position_x"],
            y=df_shot_positions_detected_events["shot_position_y"],
            mode="markers",
            name="Detected Events",
            marker=dict(size=6, opacity=0.85),
            hovertemplate="x: %{x:.2f} m<br>y: %{y:.2f} m<extra>Detected</extra>",
        )
    )

# optional second trace (if you also want to see Kinexon event positions)
if {"x in m", "y in m"}.issubset(df_shot_positions_detected_events.columns):
    fig.add_trace(
        go.Scattergl(
            x=df_shot_positions_detected_events["x in m"],
            y=df_shot_positions_detected_events["y in m"],
            mode="markers",
            name="Kinexon @ Event",
            marker=dict(size=6, symbol="x", opacity=0.9),
            hovertemplate="x: %{x:.2f} m<br>y: %{y:.2f} m<extra>Kinexon</extra>",
        )
    )

# background field (placed beneath data, full-canvas, correct scale)
if bg_source:
    fig.update_layout(
        images=[
            dict(
                source=bg_source,
                xref="x",
                yref="y",
                x=x_range[0],
                y=y_range[1],   # top
                sizex=x_range[1] - x_range[0],
                sizey=y_range[1] - y_range[0],
                sizing="stretch",
                opacity=0.55,
                layer="below",
            )
        ]
    )

# axes, title, and layout polish
fig.update_layout(
    title=dict(text="Shot Positions on Handball Field", x=0.02, xanchor="left"),
    template="plotly_white",
    dragmode="pan",
    margin=dict(l=40, r=20, t=60, b=40),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0, itemwidth=60),
)

# equal aspect ratio and tidy axes
fig.update_xaxes(
    range=list(x_range),
    constrain="range",
    scaleanchor="y",
    scaleratio=1,
    showgrid=False,
    zeroline=False,
    title_text="X Position (m)",
)
fig.update_yaxes(
    range=list(y_range),
    constrain="range",
    showgrid=False,
    zeroline=False,
    title_text="Y Position (m)",
)
import numpy as np
import plotly.graph_objects as go

# assume the two point sets are row-aligned:
# ("shot_position_x","shot_position_y") vs ("x in m","y in m")

cols_a = {"shot_position_x", "shot_position_y"}
cols_b = {"x in m", "y in m"}

if cols_a.issubset(df_shot_positions_detected_events.columns) and cols_b.issubset(df_shot_positions_detected_events.columns):
    a_x = df_shot_positions_detected_events["shot_position_x"].astype(float)
    a_y = df_shot_positions_detected_events["shot_position_y"].astype(float)
    b_x = df_shot_positions_detected_events["x in m"].astype(float)
    b_y = df_shot_positions_detected_events["y in m"].astype(float)

    # valid rows only
    valid = (~a_x.isna()) & (~a_y.isna()) & (~b_x.isna()) & (~b_y.isna())
    dx = (a_x - b_x)
    dy = (a_y - b_y)
    dist = np.hypot(dx, dy)

    sel = valid & (dist > 2.0)  # > 2 m

    # Build a lines trace: (ax, ay) -> (bx, by) with NaN separators
    xs = np.column_stack([a_x[sel].to_numpy(), b_x[sel].to_numpy(), np.full(sel.sum(), np.nan)]).ravel()
    ys = np.column_stack([a_y[sel].to_numpy(), b_y[sel].to_numpy(), np.full(sel.sum(), np.nan)]).ravel()

    fig.add_trace(
        go.Scatter(
            x=xs, y=ys,
            mode="lines",
            name="Δ > 2 m",
            line=dict(width=1.5, dash="dot"),
            hoverinfo="skip",
            showlegend=True,
        )
    )

    # Optional: mid-point markers to show the actual distance on hover
    mx = (a_x[sel].to_numpy() + b_x[sel].to_numpy()) / 2.0
    my = (a_y[sel].to_numpy() + b_y[sel].to_numpy()) / 2.0
    fig.add_trace(
        go.Scatter(
            x=mx, y=my,
            mode="markers",
            name="Δ distance",
            marker=dict(size=4, opacity=0.6),
            hovertemplate="Δ = %{customdata:.2f} m<extra></extra>",
            customdata=dist[sel].to_numpy(),
            showlegend=False,
        )
    )


fig.show()


In [ ]:
# get fixture_id for the current session_id from kinexon_positions
fixture_id = con.execute(f"SELECT fixture_id FROM kinexon_positions WHERE session_id='{session_id}' LIMIT 1").fetchone()[0]

# get match_events for that fixture_id
df_match_events = con.execute(f"SELECT * FROM match_events WHERE fixtureId='{fixture_id}'").fetchdf()
# goal events
df_goal_events = df_match_events[df_match_events['eventType'] == 'goal']
print(f'Initial length of df_goal_events: {len(df_goal_events)}')
# join league_id from table players
display(df_goal_events)
players_df = con.execute("SELECT personId, entityId, league_id FROM players").fetchdf()
df_goal_events = df_goal_events.merge(players_df, on=["personId", "entityId"], how="left")
df_goal_events = df_goal_events.merge(con.execute("SELECT personId as goalKeeperId, league_id as goalkeeper_league_id FROM players").fetchdf(), on="goalKeeperId", how="left")

# analyze how many goals in df_goal_events are also in df_merged by comparing timestamps and player_ids
# isoformat timestamps with ISO 8601 format
df_goal_events["eventTime"] = pd.to_datetime(df_goal_events["eventTime"], format='ISO8601')
df_goal_events['eventTime_ms'] = df_goal_events['eventTime'].astype('int64') // 10**6


# Function to find closest timestamp match within tolerance
def find_closest_timestamp_match(goal_timestamp_ms, tolerance_ms=30000):  # 30 second tolerance
    if df_events_shot_detected.empty or 'timestamp_ms' not in df_events_shot_detected.columns:
        return None
    
    time_diffs = abs(df_events_shot_detected['timestamp_ms'] - goal_timestamp_ms)
    if time_diffs.empty:
        return None
        
    min_diff_idx = time_diffs.idxmin()
    min_diff = time_diffs.loc[min_diff_idx]
    
    if min_diff <= tolerance_ms:
        return df_events_shot_detected.loc[min_diff_idx, 'timestamp_ms'] - goal_timestamp_ms
    else:
        return None

# find matching events in df_events_shot_detected and write time difference in ms into new column 'time_diff_ms'
df_goal_events['time_diff_ms'] = df_goal_events['eventTime_ms'].apply(find_closest_timestamp_match)

# Add matched timestamp for reference
def find_matched_timestamp(goal_timestamp_ms, tolerance_ms=30000):
    if df_events_shot_detected.empty or 'timestamp_ms' not in df_events_shot_detected.columns:
        return None
    
    time_diffs = abs(df_events_shot_detected['timestamp_ms'] - goal_timestamp_ms)
    if time_diffs.empty:
        return None
        
    min_diff_idx = time_diffs.idxmin()
    min_diff = time_diffs.loc[min_diff_idx]
    
    if min_diff <= tolerance_ms:
        return df_events_shot_detected.loc[min_diff_idx, 'timestamp_ms']
    else:
        return None

df_goal_events['matched_timestamp_ms'] = df_goal_events['eventTime_ms'].apply(find_matched_timestamp)

display(df_goal_events)

In [ ]:
# Cell 9 — Renderer: draw players, ball, and highlight shooter by league_id match
import cv2
OUT_DIR = Path("data/metadata")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FIELD_IMAGE = Path("data/metadata/handballfeld.png")
if not FIELD_IMAGE.exists():
    # Try parent path as fallback relative to notebook location
    alt = Path("..") / FIELD_IMAGE
    FIELD_IMAGE = alt if alt.exists() else Path("data/metadata/handballfeld.png")


def render_goal_locally(df_kinexon: pd.DataFrame, row_sportradar: pd.Series) -> None:
    """
    Render a visualization of the goal event based on Kinexon data and a Sportradar goal row.
    Highlights the shooter using league_id mapping (already written to players).
    """
    img = cv2.imread(str(FIELD_IMAGE))
    if img is None:
        print(f"⚠️ Could not read field image at: {FIELD_IMAGE}")
        return

    height, width = img.shape[:2]
    scale = width / 40.0  # simple meter->pixel scaling assuming 40m width

    name_event = f'event_{row_sportradar["eventId"]}'
    out_path = OUT_DIR / f"{name_event}.mp4"

    writer = cv2.VideoWriter(
        str(out_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        20,
        (width, height),
    )

    

    # Group frames by time order
    for ts, group in df_kinexon.groupby("ts"):
        img_draw = img.copy()

        for _, r in group.iterrows():
            # group id: 3 = ball? keep color coding
            gid = r.get("group id", None)
            color, radius = (0, 0, 0), 17
            if gid == 3:
                color, radius = (0, 0, 255), 15
            elif gid == 2:
                color, radius = (0, 255, 0), 10
            elif gid == 1:
                color, radius = (255, 0, 0), 10
            
            if pd.isna(r["x in m"]) or pd.isna(r["y in m"]):
                continue
            x = int(float(r["x in m"]) * scale)
            y = int(float(r["y in m"]) * scale)
            cv2.circle(img_draw, (x, y), radius, color, -1)
            # write league id above
            league_id = r.get("league id", "N/A")
            cv2.putText(img_draw, f"ID:{league_id}", (x - 10, y - radius - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
            # name above
            name = r.get("full name", "N/A")
            cv2.putText(img_draw, name, (x - 10, y - radius - 25), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
            # draw shooter highlight if league id matches
            try:
                shooter_league_id = row_sportradar.get("person_league_id", None)
                if shooter_league_id is not None and int(r.get("league id", -1)) == int(shooter_league_id):
                    cv2.circle(img_draw, (x, y), radius + 5, (255, 255, 0), 2)
            except Exception:
                pass
            # draw goalkeeper highlight if league id matches
            try:
                goalkeeper_league_id = row_sportradar.get("goalkeeper_league_id", None)
                if goalkeeper_league_id is not None and int(r.get("league id", -1)) == int(goalkeeper_league_id):
                    cv2.circle(img_draw, (x, y), radius + 5, (0, 255, 255), 2)
            except Exception:
                pass

            # write game info (name, team, type, time)
            attackType = row_sportradar.get("attackType", "unknown")
            sub_type = row_sportradar.get("subType", "unknown")
            success = row_sportradar.get("success", True)
            name_player = row_sportradar.get("personName", "Unknown Player")
            name_team_player = row_sportradar.get("teamName", "Unknown Team")
            name_goalkeeper = row_sportradar.get("goalkeeperName", "Unknown Goalkeeper")
            id_player = row_sportradar.get("person_league_id", "N/A")
            id_goalkeeper = row_sportradar.get("goalkeeper_league_id", "N/A")
            cv2.putText(img_draw, f"Player: {name_player} (ID: {id_player}) (Team: {name_team_player}) vs. Goalkeeper: {name_goalkeeper} (ID: {id_goalkeeper})", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            cv2.putText(img_draw, f"Event: Goal (Type: {attackType}, Sub: {sub_type}), Success: {success} | Time: {ts.strftime('%H:%M:%S')}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            # difference of timestamps
            cv2.putText(img_draw, f"Time Diff (ms): {row_sportradar.get('time_diff_ms', 'N/A')}", (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            # check if ts matches matched_timestamp_ms
            ts_row = row_sportradar.get('matched_timestamp_ms', -1)
            if abs(ts.value / 1e6 - ts_row) < 1e-3:
                cv2.putText(img_draw, ">> MATCHED GOAL EVENT <<", (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
                # waitkey to highlight
                cv2.waitKey(1000)
                cv2.imshow("Frame", img_draw)

        cv2.imshow("Frame", img_draw)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):  # brief wait to render
            break
        # write frame
        writer.write(cv2.resize(img_draw, (width, height)))

    writer.release()
    print(f"🎬 Saved: {out_path}")


In [ ]:
# Convert timestamps to proper datetime objects
df_positions['ts'] = pd.to_datetime(
    df_positions["ts in ms"], unit="ms", utc=True, errors="coerce"
)
df_goal_events['eventTime'] = pd.to_datetime(
    df_goal_events["eventTime"], format="ISO8601", errors="coerce"
)
# join league id from players to goals
df_goal_events = df_goal_events.merge(
    con.execute("SELECT personId, league_id AS person_league_id FROM players").df(),
    on="personId",
    how="left"
)
# df_goal_events = df_goal_events.merge(
#     con.execute("SELECT personId as goalKeeperId, league_id AS goalkeeper_league_id FROM players").df(),
#     on="goalKeeperId",
#     how="left"
# )

display(df_goal_events)

for _, row in df_goal_events.iterrows():
    print(f"Rendering goal eventId: {row['eventId']}")
    # Use proper timestamp and timedelta operations
    event_time = row["eventTime"]  # Already converted to datetime in previous cell
    df_kinexon_scene = df_positions[
        (df_positions["ts"] >= event_time - pd.Timedelta(seconds=15)) &
        (df_positions["ts"] <= event_time)
    ]
    render_goal_locally(df_kinexon_scene, row)

cv2.destroyAllWindows()